# Parte 3 – Cruce por ubigeo y análisis

En esta tercera parte unimos tres base de datos: las **lluvias por departamento**, el **conteo de decretos de emergencia por departamento** y el **listado de decretos supremos** (PCM) por intensas precipitaciones. La llave para unir las tablas será el **ubigeo departamental del INEI**, no el nombre.



In [1]:
import os
import unicodedata
import pandas as pd
import plotly.express as px

os.makedirs("datos", exist_ok=True)
pd.set_option("display.max_columns", None)

## Paso 1 — Crear `datos/ubigeos.csv`

In [18]:
ubigeos_lista = [
    ("01", "Amazonas"), ("02", "Áncash"), ("03", "Apurímac"), ("04", "Arequipa"),
    ("05", "Ayacucho"), ("06", "Cajamarca"), ("07", "Callao"), ("08", "Cusco"),
    ("09", "Huancavelica"), ("10", "Huánuco"), ("11", "Ica"), ("12", "Junín"),
    ("13", "La Libertad"), ("14", "Lambayeque"), ("15", "Lima"), ("16", "Loreto"),
    ("17", "Madre de Dios"), ("18", "Moquegua"), ("19", "Pasco"), ("20", "Piura"),
    ("21", "Puno"), ("22", "San Martín"), ("23", "Tacna"), ("24", "Tumbes"),
    ("25", "Ucayali"),
]
ubigeos_df = pd.DataFrame(ubigeos_lista, columns=["ubigeo", "departamento"])
ubigeos_df.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8")
print("Filas:", len(ubigeos_df))
ubigeos_df.head(10)

Filas: 25


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


## Paso 2 — Leer el archivo dos veces

In [19]:
# Lectura 1: normal (pandas adivina el tipo de dato)
ubi_normal = pd.read_csv("datos/ubigeos.csv")

# Lectura 2: forzando ubigeo como texto
ubi_texto = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})

print("Lectura normal  -> tipo:", ubi_normal["ubigeo"].dtype)
print("Lectura con str -> tipo:", ubi_texto["ubigeo"].dtype)

comparacion = pd.DataFrame({
    "departamento": ubi_texto["departamento"],
    "ubigeo_normal": ubi_normal["ubigeo"],
    "ubigeo_texto": ubi_texto["ubigeo"],
})
comparacion.head(10)

Lectura normal  -> tipo: int64
Lectura con str -> tipo: str


,departamento,ubigeo_normal,ubigeo_texto
0,Amazonas,1,01
1,Áncash,2,02
2,Apurímac,3,03
3,Arequipa,4,04
4,Ayacucho,5,05
5,Cajamarca,6,06
6,Callao,7,07
7,Cusco,8,08
8,Huancavelica,9,09
9,Huánuco,10,10


**¿Por qué importa?**

Con `pd.read_csv` normal, pandas ve que la columna `ubigeo` solo tiene dígitos y la convierte en **número entero**: `"01"` pasa a ser `1`, `"09"` pasa a ser `9`. Se pierde el **cero a la izquierda**.

Eso es un problema porque el ubigeo **no es una cantidad, es un código** (como un DNI o un número de teléfono): no tiene sentido sumarlo ni promediarlo, y su forma exacta importa. Si una tabla tiene `"01"` (texto) y otra tiene `1` (número), el cruce falla o se mezclan tipos; además, los ubigeos provinciales y distritales (`010101`) se construyen concatenando el departamental, y con `1` en vez de `01` se generarían códigos incorrectos. Por eso, desde ahora, **siempre leemos el ubigeo con `dtype={"ubigeo": str}`**.

## Paso 3 — Agregar el ubigeo a las dos tablas departamentales

In [4]:
ubigeos = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv")
decretos = pd.read_csv("datos/decretos_por_departamento.csv")

print("lluvias:", lluvias.shape, "| decretos:", decretos.shape)

lluvias_u = lluvias.merge(ubigeos, on="departamento", how="left")
decretos_u = decretos.merge(ubigeos, on="departamento", how="left")
lluvias_u.head()

lluvias: (25, 6) | decretos: (25, 3)


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,ubigeo
0,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,01
1,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,02
2,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,03
3,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,04
4,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,05


## Paso 4 — Verificación: ¿qué departamentos no encontraron su ubigeo?

Si algún nombre de departamento está escrito de manera distinta (sin tilde, con espacios extra, en mayúsculas, "Ancash" vs "Áncash"), el `merge` deja el `ubigeo` vacío (`NaN`).

In [5]:
def sin_ubigeo(df, nombre):
    faltan = df[df["ubigeo"].isna()]["departamento"].tolist()
    print(f"{nombre}: {len(faltan)} sin ubigeo -> {faltan}")
    return faltan

faltan_ll = sin_ubigeo(lluvias_u, "lluvias")
faltan_de = sin_ubigeo(decretos_u, "decretos")

lluvias: 0 sin ubigeo -> []
decretos: 0 sin ubigeo -> []


Para corregir cualquier diferencia de escritura de forma sistemática, creamos una **llave normalizada**: sin tildes, sin espacios sobrantes y en minúsculas. Así `"Áncash"`, `"ancash"` y `" ANCASH "` se vuelven la misma llave `"ancash"`. Repetimos el cruce con esa llave y volvemos a verificar hasta que no quede ninguno sin ubigeo.

In [6]:
def normalizar(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())

ubigeos["llave"] = ubigeos["departamento"].apply(normalizar)

def agregar_ubigeo(df):
    df = df.copy()
    df["llave"] = df["departamento"].apply(normalizar)
    df = df.merge(ubigeos[["llave", "ubigeo"]], on="llave", how="left").drop(columns="llave")
    return df

lluvias_u = agregar_ubigeo(lluvias)
decretos_u = agregar_ubigeo(decretos)

faltan_ll = sin_ubigeo(lluvias_u, "lluvias")
faltan_de = sin_ubigeo(decretos_u, "decretos")
assert not faltan_ll and not faltan_de, "Todavía hay departamentos sin ubigeo"
print("✔ Todos los departamentos tienen ubigeo en ambas tablas")

lluvias: 0 sin ubigeo -> []
decretos: 0 sin ubigeo -> []
✔ Todos los departamentos tienen ubigeo en ambas tablas


> **Nota:** en nuestros archivos los nombres ya venían bien escritos (con tildes iguales a las del INEI), así que la lista de faltantes salió vacía incluso antes de normalizar. Igual dejamos la normalización: es la protección estándar cuando se cruzan bases de fuentes distintas.

## Paso 5 — Cruzar las tablas **por `ubigeo`**

Usamos la lista oficial de 25 ubigeos como tabla base y le pegamos las otras dos con `how="left"`. Así ningún departamento puede desaparecer aunque falte en alguna fuente.

In [22]:
tabla_final = (
    ubigeos[["ubigeo", "departamento"]]
    .merge(lluvias_u.drop(columns="departamento"), on="ubigeo", how="left")
    .merge(decretos_u.drop(columns="departamento"), on="ubigeo", how="left")
)
tabla_final

,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
1,02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
2,03,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,1,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,4,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,5,1
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0,4,0
6,07,Callao,Callao,-12.05162,-77.13452,55.5,0,1,0
7,08,Cusco,Cusco,-13.53188,-71.96701,714.0,1,2,0
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3,3,0
9,10,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2,4,0


## Paso 6 — Verificación: 25 filas y ceros donde no hubo declaratorias

In [9]:
tabla_final[["declaratorias", "prorrogas"]] = (
    tabla_final[["declaratorias", "prorrogas"]].fillna(0).astype(int)
)

print("Filas:", len(tabla_final))
assert len(tabla_final) == 25, "La tabla final no tiene 25 filas"
assert tabla_final["ubigeo"].is_unique, "Hay ubigeos repetidos"

print("\nDepartamentos con 0 declaratorias:")
tabla_final[tabla_final["declaratorias"] == 0][["ubigeo", "departamento", "lluvia_total_mm", "declaratorias"]]

Filas: 25

Departamentos con 0 declaratorias:


,ubigeo,departamento,lluvia_total_mm,declaratorias
16,17,Madre de Dios,1248.4,0


## Paso 7 — Guardar `datos/tabla_final.csv` con ubigeo de 2 dígitos

In [10]:
tabla_final["ubigeo"] = tabla_final["ubigeo"].astype(str).str.zfill(2)
tabla_final.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8")

# Comprobación: leemos el archivo como texto plano (primeras líneas)
with open("datos/tabla_final.csv", encoding="utf-8") as f:
    for _ in range(4):
        print(f.readline().rstrip())

# Y lo releemos con pandas forzando str
control = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str})
assert control["ubigeo"].str.len().eq(2).all()
print("\n✔ ubigeo guardado como texto de 2 dígitos:", control["ubigeo"].head(3).tolist())

ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
03,Apurímac,Abancay,-13.63426,-72.8838,726.2,3,1,0

✔ ubigeo guardado como texto de 2 dígitos: ['01', '02', '03']


In [11]:
tabla_final

,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,414.6,3,3,1
1,02,Áncash,Huaraz,-9.52614,-77.52869,1145.8,9,6,1
2,03,Apurímac,Abancay,-13.63426,-72.88380,726.2,3,1,0
3,04,Arequipa,Arequipa,-16.39899,-71.53747,362.0,1,4,0
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,498.4,2,5,1
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,796.6,0,4,0
6,07,Callao,Callao,-12.05162,-77.13452,55.5,0,1,0
7,08,Cusco,Cusco,-13.53188,-71.96701,714.0,1,2,0
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,817.8,3,3,0
9,10,Huánuco,Huánuco,-9.92882,-76.23989,524.3,2,4,0


## Paso 8 — Gráficos con Plotly

In [31]:
# 1) Nota al pie (periodo y fuente)
NOTA = (
    "<b>Periodo:</b> 1 de enero al 31 de mayo de 2023 "
    "(fecha de publicación de los decretos y registro de lluvias).<br>"
    "<b>Fuente:</b> Decretos supremos de la PCM (gob.pe) y datos de lluvia por departamento."
)

def agregar_nota(fig):
    fig.add_annotation(
        text=NOTA,
        xref="paper", yref="paper",
        x=0, y=-0.2, xanchor="left", yanchor="top",
        showarrow=False, align="left",
        font=dict(size=11, color="gray"),
    )
    fig.update_layout(margin=dict(t=110, b=120))  # espacio arriba (título) y abajo (nota)
    return fig

# 2) Gráfico de dispersión
fig_disp = px.scatter(
    tabla_final,
    x="lluvia_total_mm",
    y="declaratorias",
    text="departamento",
    hover_data=["ubigeo", "dias_lluvia_fuerte", "prorrogas"],
    title="<b>Más lluvia no significó más emergencias</b>"
          "<br><span style='font-size:15px'>Lluvia acumulada vs. declaratorias por departamento</span>",
    labels={"lluvia_total_mm": "Lluvia acumulada (mm)", "declaratorias": "N.° de declaratorias"},
)
fig_disp.update_traces(textposition="top center", marker=dict(size=10), cliponaxis=False)
fig_disp.update_layout(
    height=650,
    template="plotly_white",
    title_x=0.5,
    title_xanchor="center",
    title_font_size=26,  # título más grande
    yaxis_range=[-0.5, tabla_final["declaratorias"].max() + 1],
)

# 3) Agregar la nota y recién mostrar
agregar_nota(fig_disp)
fig_disp.show()

In [32]:
top10 = tabla_final.sort_values(["declaratorias", "departamento"], ascending=[False, True]).head(10)

fig_bar = px.bar(
    top10,
    x="departamento",
    y="declaratorias",
    text="declaratorias",
    title="<b>Lima y Áncash concentran más declaratorias</b>"
          "<br><span style='font-size:15px'>Top 10 departamentos por n.° de declaratorias</span>",
    labels={"departamento": "", "declaratorias": "N.° de declaratorias"},
)
fig_bar.update_traces(textposition="outside", cliponaxis=False)
fig_bar.update_layout(
    height=580,
    template="plotly_white",
    title_x=0.5,
    title_xanchor="center",
    title_font_size=26,  # título más grande
    yaxis_range=[0, top10["declaratorias"].max() + 1],
)

# Agregar la nota (periodo y fuente) y recién mostrar
agregar_nota(fig_bar)
fig_bar.update_annotations(y=-0.25)  # un poco más abajo para no chocar con los nombres
fig_bar.show()

> En el puesto 10 hay un **empate**: varios departamentos tienen 3 declaratorias. Para que el top 10 sea reproducible, desempatamos por orden alfabético.

## Paso 9 — Preguntas

### i. Top 3 en declaratorias vs. top 3 en lluvia

In [14]:
cols = ["departamento", "declaratorias", "lluvia_total_mm"]
top_decl = tabla_final.sort_values("declaratorias", ascending=False)[cols].head(4)
top_lluvia = tabla_final.sort_values("lluvia_total_mm", ascending=False)[cols].head(3)
print("Más declaratorias:"); display(top_decl)
print("Más lluvia:"); display(top_lluvia)

tabla_final["rank_lluvia"] = tabla_final["lluvia_total_mm"].rank(ascending=False).astype(int)
tabla_final[tabla_final["departamento"].isin(top_decl["departamento"])][["departamento", "declaratorias", "rank_lluvia"]]

Más declaratorias:


,departamento,declaratorias,lluvia_total_mm
14,Lima,7,55.5
1,Áncash,6,1145.8
4,Ayacucho,5,498.4
12,La Libertad,5,177.2


Más lluvia:


,departamento,declaratorias,lluvia_total_mm
15,Loreto,1,1604.6
24,Ucayali,3,1316.0
16,Madre de Dios,0,1248.4


,departamento,declaratorias,rank_lluvia
1,Áncash,6,4
4,Ayacucho,5,16
12,La Libertad,5,20
14,Lima,7,23


**Respuesta.** Los departamentos con más declaratorias fueron **Lima (7)**, **Áncash (6)** y, empatados en el tercer lugar, **Ayacucho y La Libertad (5 cada uno)**.

**No coinciden** con los que más llovió, que fueron **Loreto (1 604,6 mm)**, **Ucayali (1 316,0 mm)** y **Madre de Dios (1 248,4 mm)**. Esos tres tuvieron apenas 1, 3 y 0 declaratorias. El caso más llamativo es Lima: es el que más declaratorias tiene, pero su estación registró solo 55,5 mm (puesto 23 de 25 en lluvia). Solo Áncash aparece arriba en ambas listas (2.º en declaratorias y 4.º en lluvia, con 1 145,8 mm).

### ii. ¿Mucha lluvia y pocas declaratorias, o al revés?

In [15]:
corr_p = tabla_final["lluvia_total_mm"].corr(tabla_final["declaratorias"])
corr_s = tabla_final["lluvia_total_mm"].corr(tabla_final["declaratorias"], method="spearman")
print(f"Correlación de Pearson:  {corr_p:.2f}")
print(f"Correlación de Spearman: {corr_s:.2f}")

med_ll = tabla_final["lluvia_total_mm"].median()
med_de = tabla_final["declaratorias"].median()
print(f"\nMediana lluvia: {med_ll} mm | Mediana declaratorias: {med_de}")

print("\nMucha lluvia (> mediana) y pocas declaratorias (≤ 1):")
display(tabla_final[(tabla_final["lluvia_total_mm"] > med_ll) & (tabla_final["declaratorias"] <= 1)][cols])
print("Poca lluvia (< mediana) y muchas declaratorias (≥ 4):")
display(tabla_final[(tabla_final["lluvia_total_mm"] < med_ll) & (tabla_final["declaratorias"] >= 4)][cols])

Correlación de Pearson:  -0.24
Correlación de Spearman: -0.17

Mediana lluvia: 714.0 mm | Mediana declaratorias: 3.0

Mucha lluvia (> mediana) y pocas declaratorias (≤ 1):


,departamento,declaratorias,lluvia_total_mm
2,Apurímac,1,726.2
15,Loreto,1,1604.6
16,Madre de Dios,0,1248.4


Poca lluvia (< mediana) y muchas declaratorias (≥ 4):


,departamento,declaratorias,lluvia_total_mm
3,Arequipa,4,362.0
4,Ayacucho,5,498.4
9,Huánuco,4,524.3
12,La Libertad,5,177.2
14,Lima,7,55.5


**Respuesta.** Sí, en ambos sentidos. La correlación entre lluvia y declaratorias es incluso **ligeramente negativa** (≈ −0,24), es decir, no se cumple que "más lluvia = más emergencias".

- **Mucha lluvia, pocas o ninguna declaratoria:** Loreto (1 604,6 mm; 1 declaratoria), Madre de Dios (1 248,4 mm; 0), Apurímac (726,2 mm; 1) y, cerca de la mediana, Puno (660,4 mm; 1).
- **Poca lluvia, muchas declaratorias:** Lima (55,5 mm; 7), La Libertad (177,2 mm; 5), Ayacucho (498,4 mm; 5), Arequipa (362,0 mm; 4) y Huánuco (524,3 mm; 4).

**Razones posibles:**
1. **Lo que es "normal" depende del lugar.** En la Amazonía (Loreto, Madre de Dios, Ucayali) llueve mucho todos los años; la infraestructura y la población están más adaptadas. En la costa, donde casi no llueve, unos pocos milímetros (o un evento puntual como el ciclón Yaku de marzo de 2023) pueden causar huaicos, inundaciones y colapso de viviendas y drenajes.
2. **El dato de lluvia es de una sola estación, en la capital.** La lluvia de "Lima" se mide en la ciudad de Lima, pero varias declaratorias fueron para distritos de sierra (por ejemplo, Yauyos), donde sí llovió fuerte. Un punto no representa a todo el departamento.
3. **Exposición y vulnerabilidad:** las emergencias dependen de cuánta gente, viviendas y carreteras hay en zonas de quebrada o de inundación, no solo de cuánto llueve.
4. **Factores institucionales:** declarar emergencia requiere que el gobierno regional/local evalúe daños y lo solicite; la capacidad técnica, la cercanía a Lima y la visibilidad mediática influyen en cuántas solicitudes se tramitan.

### iii. Peligro inminente vs. impacto de daños

In [16]:
dec = pd.read_csv("datos/decretos_lluvias.csv")
print("Decretos en la base:", len(dec))
display(pd.crosstab(dec["tipo"], dec["motivo"], margins=True))

declara = dec[dec["tipo"] == "declara"]
pct_decretos = (declara["motivo"].value_counts(normalize=True) * 100).round(1)
print("\n% de los DECRETOS que declaran emergencia (n =", len(declara), "):")
print(pct_decretos.to_string())

Decretos en la base: 21


motivo,impacto de daños,otro,peligro inminente,All
tipo,,,,
declara,14,1,3,18
prorroga,2,1,0,3
All,16,2,3,21



% de los DECRETOS que declaran emergencia (n = 18 ):
motivo
impacto de daños     77.8
peligro inminente    16.7
otro                  5.6


Un mismo decreto puede incluir varios departamentos. Como nuestra tabla cuenta **declaratorias por departamento** (73 en total), también calculamos el porcentaje ponderado: cuántas "declaratorias departamentales" aporta cada tipo de motivo.

In [17]:
nombres = ubigeos["departamento"].tolist()
dec["n_departamentos"] = dec["titulo"].apply(lambda t: sum(d in t for d in nombres))
declara = dec[dec["tipo"] == "declara"]

por_depto = declara.groupby("motivo")["n_departamentos"].sum()
print("Total declaratorias departamentales:", por_depto.sum(),
      "| coincide con la tabla:", tabla_final["declaratorias"].sum())
(por_depto / por_depto.sum() * 100).round(1).rename("% por departamento").to_frame()

Total declaratorias departamentales: 73 | coincide con la tabla: 73


,% por departamento
motivo,
impacto de daños,45.2
otro,4.1
peligro inminente,50.7


**Respuesta.** De los **18 decretos que declaran** emergencia en la temporada 2023:

| Motivo | Decretos | % de decretos | Declaratorias departamentales | % ponderado |
|---|---|---|---|---|
| Peligro inminente (antes del daño) | 3 | **16,7 %** | 37 | 50,7 % |
| Impacto de daños (después) | 14 | **77,8 %** | 33 | 45,2 % |
| Otro (desastre de gran magnitud) | 1 | 5,6 % | 3 | 4,1 % |

*(Las 3 prórrogas no se cuentan como nuevas declaratorias.)*

**¿Qué dice esto sobre la gestión del riesgo?** La gran mayoría de los decretos (≈ 78 %, o ≈ 83 % si sumamos el "desastre de gran magnitud", que también es posterior al daño) se emitieron **después** de que el daño ya ocurrió. La gestión fue principalmente **reactiva**. Hubo tres decretos preventivos, pero fueron muy amplios (cubren de 7 a 16 departamentos cada uno) y concentrados en enero y marzo; por eso, al contar por departamento, la prevención parece pesar la mitad. Pero en número de decisiones, el Estado actuó sobre todo respondiendo a daños puntuales, distrito por distrito. Esto sugiere que hay espacio para fortalecer la alerta temprana y usar más la declaratoria por peligro inminente, que permite intervenir antes de que caigan los huaicos.

### iv. Limitaciones del análisis

1. **La lluvia de una estación no representa al departamento.** Las coordenadas corresponden a la capital; en departamentos con costa, sierra y selva (Lima, Áncash, Arequipa) la lluvia varía enormemente. Lima y Callao incluso comparten el mismo valor (55,5 mm).
2. **Contar decretos no mide la magnitud del daño.** Un decreto puede abarcar 1 distrito o 100; no sabemos cuántas personas fueron afectadas, viviendas destruidas o pérdidas económicas.
3. **El conteo depende de cómo se asignan los decretos a departamentos.** Los decretos multidepartamentales cuentan una vez por cada departamento mencionado; además, solo tenemos 21 decretos, y pudo haber otros (por ejemplo, por huaicos o por "otros" motivos) no incluidos en la base.
4. **No hay control por exposición ni vulnerabilidad.** No consideramos población, tipo de vivienda, pendiente del terreno ni infraestructura; tampoco el promedio histórico de lluvia de cada zona (anomalía), que es lo que realmente indica una lluvia "extrema".
5. **Correlación no implica causalidad y la muestra es pequeña (25 departamentos, una sola temporada).** Las declaratorias también dependen de decisiones políticas y administrativas.

### v. Conclusión

En la temporada de lluvias 2023, **los departamentos donde más llovió no fueron los que tuvieron más declaratorias de emergencia**: Loreto, Ucayali y Madre de Dios encabezan la lluvia con pocas o ninguna declaratoria, mientras que Lima, Áncash, Ayacucho y La Libertad encabezan las declaratorias con lluvias moderadas o bajas. La correlación entre ambas variables es débil e incluso ligeramente negativa (≈ −0,24). Esto indica que la ocurrencia de emergencias depende menos del total de lluvia y más de la **vulnerabilidad y exposición** de cada territorio (la costa y la sierra occidental no están preparadas para lluvias intensas), así como de eventos puntuales como el ciclón Yaku. Además, la mayoría de decretos se emitió **por impacto de daños**, lo que muestra una gestión del riesgo predominantemente reactiva. Para confirmar estas conclusiones se necesitaría lluvia medida en todo el territorio y datos de daños, no solo el número de decretos.